# YB-Mixer — GPU benchmark (Kaggle), v4: fair baselines + multi-seed

Spectral **YB-Mixer** vs **fairly-tuned** baselines in a controlled scaffold (only the token mixer changes):
`s4dlin` (S4D with **HiPPO/S4D-Lin init** — the *fair* SSM), `lru` (Linear Recurrent Unit),
`fnet` (fixed-FFT cousin), `transformer`, real `mamba` (selective, if `mamba-ssm` installs),
plus `s4d` (untuned, kept as an init-ablation).

**Reports mean ± std over ≥3 seeds** and **inference throughput** (latency evidence).

### Before running
1. **Settings → Accelerator → GPU** (T4/P100).  2. **Internet → On** (downloads + mamba-ssm).
3. Run top to bottom. Watch the **budget note** before Phase A — multi-seed is expensive.


## 1 · Environment  (also tries to install the real Mamba kernel)

In [ ]:
import torch, sys, platform, subprocess
print('python', platform.python_version(), '| torch', torch.__version__)
print('CUDA:', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
try:
    import datasets  # noqa
except Exception:
    subprocess.run([sys.executable,'-m','pip','install','-q','datasets'])
# Real selective Mamba (best-effort; needs a matching CUDA toolchain). Falls back if it fails.
try:
    import mamba_ssm  # noqa
    print('mamba-ssm already present')
except Exception:
    subprocess.run([sys.executable,'-m','pip','install','-q','mamba-ssm'])
try:
    import mamba_ssm; print('REAL Mamba kernel available:', True)
except Exception as e:
    print('REAL Mamba NOT available -> labeled gated-SSM fallback will be used. (', type(e).__name__, ')')


## 2 · Models  (`%%writefile models.py`)

In [ ]:
%%writefile models.py
"""
Models for the YB-Mixer GPU harness.

A single shared block scaffold (pre-norm: x = x + mixer(norm(x)); x = x + mlp(norm(x)))
is used for every architecture, so comparisons differ ONLY in the token mixer:

  - 'yb'        : Spectral YB-Mixer (orthogonal, phase-only)         <- our model
  - 'yb_relaxed': Spectral mixer with learnable magnitude (NOT orthogonal; a bidirectional
                  global-conv / linear-SSM ablation that quantifies the cost of orthogonality)
  - 'transformer': bidirectional multi-head self-attention
  - 's4d'       : diagonal complex SSM (S4D-style), bidirectional, via FFT convolution
  - 'mamba'     : real Mamba block if `mamba_ssm` is importable; else a gated diagonal SSM

All spectral / phase profiles are parameterized as functions of the NORMALIZED frequency
f = m / L, so the mixers are length-agnostic (key for length generalization).
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------- token mixers -----------------------------

class SpectralYBMixer(nn.Module):
    """Bidirectional orthogonal token mixer: per-channel unit-magnitude spectral multiplier.
    multiplier(m, c) = exp( i * phi_c(f_m) )  [+ log-magnitude if not orthogonal],
    with phi_c(f) = sum_k W[c,k] sin(2 pi k f) + V[c,k] (1 - cos(2 pi k f)),  f = m/L.
    """
    def __init__(self, dim, n_basis=24, orthogonal=True, init=0.05):
        super().__init__()
        self.dim, self.n_basis, self.orthogonal = dim, n_basis, orthogonal
        self.Wp = nn.Parameter(init * torch.randn(dim, n_basis))   # phase basis (sin)
        self.Vp = nn.Parameter(init * torch.randn(dim, n_basis))   # phase basis (1-cos)
        if not orthogonal:
            self.Wm = nn.Parameter(init * torch.randn(dim, n_basis))   # log-mag basis
            self.Vm = nn.Parameter(init * torch.randn(dim, n_basis))
        self.scale = nn.Parameter(torch.ones(()))
        self.register_buffer("K", torch.arange(1, n_basis + 1).float())

    def _profiles(self, M, L, device, dtype):
        f = torch.arange(M, device=device, dtype=dtype) / L          # (M,)
        ang = 2 * math.pi * self.K.to(dtype)[None, :] * f[:, None]   # (M, n_basis)
        s, c = torch.sin(ang), 1.0 - torch.cos(ang)                  # (M, n_basis)
        phase = (s @ self.Wp.t().to(dtype) + c @ self.Vp.t().to(dtype)) * self.scale  # (M, dim)
        if self.orthogonal:
            logmag = torch.zeros_like(phase)
        else:
            logmag = -F.softplus(s @ self.Wm.t().to(dtype) + c @ self.Vm.t().to(dtype))  # <=0
        return phase, logmag

    def forward(self, x):                       # x: (B, L, C)
        B, L, C = x.shape
        # entire spectral op in float32 with autocast OFF (avoids Half/ComplexHalf under AMP)
        with torch.autocast(device_type=x.device.type, enabled=False):
            x32 = x.float()
            Xf = torch.fft.rfft(x32, dim=1)                              # (B, M, C) complex64
            M = Xf.shape[1]
            phase, logmag = self._profiles(M, L, x.device, torch.float32)   # (M, C) float32
            mult = torch.exp(torch.complex(logmag, phase))              # complex64
            Xf = Xf * mult.unsqueeze(0)
            y = torch.fft.irfft(Xf, n=L, dim=1)                         # (B, L, C) float32
        return y.to(x.dtype)


class AttentionMixer(nn.Module):
    def __init__(self, dim, n_heads=4, dropout=0.0):
        super().__init__()
        self.mha = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)

    def forward(self, x):
        y, _ = self.mha(x, x, x, need_weights=False)
        return y


class S4DMixer(nn.Module):
    """Bidirectional diagonal complex SSM via FFT convolution (S4D-style)."""
    def __init__(self, dim, bidirectional=True):
        super().__init__()
        self.dim, self.bi = dim, bidirectional
        self.log_a = nn.Parameter(torch.log(0.5 + torch.rand(dim)))   # decay rate (>0)
        self.theta = nn.Parameter(math.pi * torch.rand(dim))          # frequency
        self.Bc = nn.Parameter(0.5 * torch.randn(dim))
        self.Cc = nn.Parameter(0.5 * torch.randn(dim))
        self.D = nn.Parameter(torch.zeros(dim))
        if bidirectional:
            self.Bc2 = nn.Parameter(0.5 * torch.randn(dim))
            self.Cc2 = nn.Parameter(0.5 * torch.randn(dim))

    def _kernel(self, L, Bc, Cc, device):
        t = torch.arange(L, device=device, dtype=torch.float32)         # (L,)
        a = torch.exp(self.log_a).clamp(max=20.0)                       # (C,)
        lam_mag = torch.exp(-a)                                         # |lambda| < 1
        ang = self.theta[None, :] * t[:, None]                          # (L, C)
        # Re( B*C* lambda^t ) with lambda = lam_mag e^{i theta}
        k = (Bc * Cc)[None, :] * (lam_mag[None, :] ** t[:, None]) * torch.cos(ang)
        return k                                                        # (L, C)

    def _causal_conv(self, x, k):                # x:(B,L,C) k:(L,C)
        B, L, C = x.shape
        n = 2 * L
        Xf = torch.fft.rfft(x.float(), n=n, dim=1)
        Kf = torch.fft.rfft(k.float(), n=n, dim=0).unsqueeze(0)        # (1, n//2+1, C)
        y = torch.fft.irfft(Xf * Kf, n=n, dim=1)[:, :L, :]
        return y

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            xf = x.float()
            L = xf.shape[1]
            k = self._kernel(L, self.Bc, self.Cc, xf.device)
            y = self._causal_conv(xf, k)
            if self.bi:
                k2 = self._kernel(L, self.Bc2, self.Cc2, xf.device)
                y = y + torch.flip(self._causal_conv(torch.flip(xf, [1]), k2), [1])
            y = y + self.D[None, None, :] * xf
        return y.to(x.dtype)


class MambaMixer(nn.Module):
    """Real Mamba block if available, else a gated diagonal SSM fallback (labeled)."""
    def __init__(self, dim, bidirectional=True):
        super().__init__()
        self.using_real = False
        try:
            from mamba_ssm import Mamba
            self.m = Mamba(d_model=dim)
            self.using_real = True
        except Exception:
            self.ssm = S4DMixer(dim, bidirectional=bidirectional)
            self.gate = nn.Linear(dim, dim)

    def forward(self, x):
        if self.using_real:
            return self.m(x)
        return self.ssm(x) * torch.sigmoid(self.gate(x))   # input-dependent gate (selective-ish)



class FNetMixer(nn.Module):
    """FNet token mixing (Lee-Thorp et al.): parameter-free 2D FFT, keep the real part.
    The closest cousin to the spectral YB-Mixer; isolates the value of a *learned orthogonal*
    spectral map over a *fixed* FFT."""
    def __init__(self, dim):
        super().__init__()
    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            xf = x.float()
            y = torch.fft.fft(torch.fft.fft(xf, dim=-1), dim=1).real
        return y.to(x.dtype)


class LRUMixer(nn.Module):
    """Linear Recurrent Unit (Orvieto et al. 2023): diagonal complex linear recurrence with
    stable magnitude/phase parameterization and gamma normalization. One complex state per
    channel; bidirectional via forward+backward FFT convolution for classification."""
    def __init__(self, dim, r_min=0.9, r_max=0.999, max_phase=6.283, bidirectional=True):
        super().__init__()
        self.bi = bidirectional
        u1 = torch.rand(dim); u2 = torch.rand(dim)
        self.nu_log = nn.Parameter(torch.log(-0.5 * torch.log(u1 * (r_max**2 - r_min**2) + r_min**2)))
        self.theta_log = nn.Parameter(torch.log(max_phase * u2))
        self.B_re = nn.Parameter(torch.randn(dim) / dim**0.5); self.B_im = nn.Parameter(torch.randn(dim) / dim**0.5)
        self.C_re = nn.Parameter(torch.randn(dim) / dim**0.5); self.C_im = nn.Parameter(torch.randn(dim) / dim**0.5)
        self.D = nn.Parameter(torch.zeros(dim))
        if bidirectional:
            self.C_re2 = nn.Parameter(torch.randn(dim) / dim**0.5); self.C_im2 = nn.Parameter(torch.randn(dim) / dim**0.5)

    def _kernel(self, L, device, C_re, C_im):
        nu = torch.exp(self.nu_log); mag = torch.exp(-nu)                  # |lambda| in (0,1)
        phase = torch.exp(self.theta_log)
        lam = torch.complex(mag * torch.cos(phase), mag * torch.sin(phase))   # (C,)
        gamma = torch.sqrt(torch.clamp(1 - mag**2, min=1e-6))                  # normalization
        B = torch.complex(self.B_re, self.B_im); C = torch.complex(C_re, C_im)
        coeff = C * gamma * B                                                  # (C,)
        t = torch.arange(L, device=device, dtype=torch.float32)
        lam_pow = lam[None, :] ** t[:, None]                                   # (L, C)
        return (coeff[None, :] * lam_pow).real                                 # (L, C)

    def _conv(self, x, k):
        L = x.shape[1]; n = 2 * L
        Xf = torch.fft.rfft(x.float(), n=n, dim=1)
        Kf = torch.fft.rfft(k, n=n, dim=0).unsqueeze(0)
        return torch.fft.irfft(Xf * Kf, n=n, dim=1)[:, :L, :]

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            xf = x.float(); L = xf.shape[1]
            y = self._conv(xf, self._kernel(L, xf.device, self.C_re, self.C_im))
            if self.bi:
                k2 = self._kernel(L, xf.device, self.C_re2, self.C_im2)
                y = y + torch.flip(self._conv(torch.flip(xf, [1]), k2), [1])
            y = y + self.D[None, None, :] * xf
        return y.to(x.dtype)


class S4DLinMixer(nn.Module):
    """Properly initialized S4D (Gu et al., 'On the Parameterization and Initialization of
    Diagonal SSMs'): N complex states per channel with the S4D-Lin/HiPPO init A_n=-1/2+i*pi*n,
    log-spaced timescales dt. Bidirectional via forward+backward FFT convolution. This is the
    *fair* SSM baseline."""
    def __init__(self, dim, n_states=64, dt_min=1e-3, dt_max=1e-1, bidirectional=True):
        super().__init__()
        self.bi = bidirectional; H, N = dim, n_states
        log_dt = torch.rand(H) * (torch.log(torch.tensor(dt_max)) - torch.log(torch.tensor(dt_min))) \
                 + torch.log(torch.tensor(dt_min))
        self.log_dt = nn.Parameter(log_dt)                                  # (H,)
        n = torch.arange(N).float()
        # parameterize Re(A) = -exp(A_re_log) < 0  => |discrete pole| < 1 ALWAYS (stable).
        self.A_re_log = nn.Parameter(torch.log(0.5 * torch.ones(H, N)))     # init Re(A) = -0.5 (S4D-Lin)
        self.A_im = nn.Parameter((math.pi * n)[None, :].repeat(H, 1).clone())  # S4D-Lin imag part
        self.C_re = nn.Parameter(torch.randn(H, N) / N**0.5); self.C_im = nn.Parameter(torch.randn(H, N) / N**0.5)
        self.D = nn.Parameter(torch.zeros(H))
        if bidirectional:
            self.C_re2 = nn.Parameter(torch.randn(H, N) / N**0.5); self.C_im2 = nn.Parameter(torch.randn(H, N) / N**0.5)

    def _kernel(self, L, device, C_re, C_im):
        dt = torch.exp(self.log_dt)[:, None]                                 # (H,1) > 0
        A = torch.complex(-torch.exp(self.A_re_log), self.A_im)              # (H,N) with Re(A) < 0
        dtA = dt * A                                                        # (H,N) complex, Re < 0
        C = torch.complex(C_re, C_im)                                       # (H,N)
        t = torch.arange(L, device=device, dtype=torch.float32)
        # lambda^t = exp(t * dt * A); magnitude = exp(t*Re(dtA)) <= 1  -> no explosion
        lam_pow = torch.exp(t[:, None, None] * dtA[None])                    # (L,H,N) complex
        k = torch.einsum('lhn,hn->lh', lam_pow, C).real                      # (L,H)
        return k

    def _conv(self, x, k):
        L = x.shape[1]; n = 2 * L
        Xf = torch.fft.rfft(x.float(), n=n, dim=1)
        Kf = torch.fft.rfft(k, n=n, dim=0).unsqueeze(0)
        return torch.fft.irfft(Xf * Kf, n=n, dim=1)[:, :L, :]

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            xf = x.float(); L = xf.shape[1]
            y = self._conv(xf, self._kernel(L, xf.device, self.C_re, self.C_im))
            if self.bi:
                k2 = self._kernel(L, xf.device, self.C_re2, self.C_im2)
                y = y + torch.flip(self._conv(torch.flip(xf, [1]), k2), [1])
            y = y + self.D[None, None, :] * xf
        return y.to(x.dtype)


# ----------------------------- block + model -----------------------------

class MLP(nn.Module):
    def __init__(self, dim, ratio=2, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, ratio * dim), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(ratio * dim, dim))

    def forward(self, x):
        return self.net(x)


class RGLRUMixer(nn.Module):
    """Real-Gated Linear Recurrent Unit (Hawk/Griffin, De et al. 2024).
    Input-dependent gates make the recurrence weight a_t = sigma(Lambda)^(c*r_t) data-dependent,
    so this CANNOT use the FFT-conv fast path; it is computed with a stable log-space sequential
    scan. Bidirectional (forward + backward) for the classification scaffold.
    NOTE: sequential scan -> slowest mixer; faithful to eq. (1)-(4)."""
    def __init__(self, dim, c=8.0, bidirectional=True):
        super().__init__()
        self.bi = bidirectional; self.c = c
        self.W_a = nn.Linear(dim, dim); self.W_x = nn.Linear(dim, dim)
        self.Lambda = nn.Parameter(self._init_lambda(dim, c))
        if bidirectional:
            self.W_a2 = nn.Linear(dim, dim); self.W_x2 = nn.Linear(dim, dim)
            self.Lambda2 = nn.Parameter(self._init_lambda(dim, c))

    @staticmethod
    def _init_lambda(dim, c):
        # init so that a^c ~ U(0.9, 0.999) at start (a = sigmoid(Lambda)), as in Griffin
        ac = torch.empty(dim).uniform_(0.9, 0.999); a = ac ** (1.0 / c)
        return torch.log(a / (1 - a))                                   # logit(a)

    def _scan(self, x, W_a, W_x, Lambda):
        r = torch.sigmoid(W_a(x)); i = torch.sigmoid(W_x(x))            # gates (B,L,C)
        log_a = self.c * r * F.logsigmoid(Lambda)[None, None, :]        # log a_t <= 0  (stable)
        a = torch.exp(log_a)
        b = torch.sqrt(torch.clamp(1 - a * a, min=1e-6)) * (i * x)      # input term
        B, L, C = x.shape; h = x.new_zeros(B, C); outs = []
        for t in range(L):
            h = a[:, t] * h + b[:, t]; outs.append(h)
        return torch.stack(outs, dim=1)

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            xf = x.float()
            y = self._scan(xf, self.W_a, self.W_x, self.Lambda)
            if self.bi:
                y = y + torch.flip(self._scan(torch.flip(xf, [1]), self.W_a2, self.W_x2, self.Lambda2), [1])
        return y.to(x.dtype)


class SCORNNMixer(nn.Module):
    """Scaled Cayley Orthogonal RNN (scoRNN, Helfrich et al. 2018).
    Recurrent matrix W = (I+A)^{-1}(I-A)D with A skew-symmetric and D diagonal +/-1 (rho = dim//2),
    so W is orthogonal to machine precision. Nonlinear (modReLU) sequential RNN -> no conv/scan
    fast path. This is the canonical orthogonal-RNN baseline for our orthogonality claim.
    NOTE: sequential + dense recurrence -> slow."""
    def __init__(self, dim, rho=None, bidirectional=True):
        super().__init__()
        self.bi = bidirectional; self.dim = dim
        rho = dim // 2 if rho is None else rho
        self._make_dir("f", dim, rho)
        if bidirectional: self._make_dir("b", dim, rho)

    def _make_dir(self, tag, dim, rho):
        setattr(self, f"A_{tag}", nn.Parameter(0.05 * torch.randn(dim, dim)))
        setattr(self, f"U_{tag}", nn.Linear(dim, dim, bias=False))
        setattr(self, f"bias_{tag}", nn.Parameter(torch.zeros(dim)))
        D = torch.ones(dim); D[:rho] = -1.0
        self.register_buffer(f"D_{tag}", D)

    def _W(self, tag):
        A = getattr(self, f"A_{tag}"); A = A - A.t()                    # skew-symmetric
        I = torch.eye(self.dim, device=A.device, dtype=A.dtype)
        return torch.linalg.solve(I + A, I - A) * getattr(self, f"D_{tag}")[None, :]   # (I+A)^-1(I-A) D

    @staticmethod
    def _modrelu(z, b):
        return torch.sign(z) * F.relu(torch.abs(z) + b)                 # real modReLU

    def _scan(self, x, tag):
        W = self._W(tag); U = getattr(self, f"U_{tag}"); b = getattr(self, f"bias_{tag}")
        B, L, C = x.shape; h = x.new_zeros(B, C); Ux = U(x); outs = []
        for t in range(L):
            h = self._modrelu(Ux[:, t] + h @ W.t(), b); outs.append(h)
        return torch.stack(outs, dim=1)

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            xf = x.float()
            y = self._scan(xf, "f")
            if self.bi: y = y + torch.flip(self._scan(torch.flip(xf, [1]), "b"), [1])
        return y.to(x.dtype)


class RWKVMixer(nn.Module):
    """RWKV-4 time-mixing (Peng et al. 2023): token-shift + R/K/V, with the numerically stable
    WKV decay-scan (running-max). Bidirectional for classification.
    NOTE: decay scan is sequential -> slow. (RWKV-4 variant; channel-mix is handled by the
    shared scaffold MLP, so this module is the time-mixing operator.)"""
    def __init__(self, dim, bidirectional=True):
        super().__init__()
        self.bi = bidirectional
        self._make_dir("f", dim)
        if bidirectional: self._make_dir("b", dim)

    def _make_dir(self, tag, dim):
        setattr(self, f"time_decay_{tag}", nn.Parameter(torch.randn(dim) * 0.1))   # w = -exp(.)
        setattr(self, f"time_first_{tag}", nn.Parameter(torch.randn(dim) * 0.1))   # bonus u
        for nm in ("mix_k", "mix_v", "mix_r"):
            setattr(self, f"{nm}_{tag}", nn.Parameter(torch.rand(dim)))
        setattr(self, f"Wk_{tag}", nn.Linear(dim, dim, bias=False))
        setattr(self, f"Wv_{tag}", nn.Linear(dim, dim, bias=False))
        setattr(self, f"Wr_{tag}", nn.Linear(dim, dim, bias=False))
        setattr(self, f"Wo_{tag}", nn.Linear(dim, dim, bias=False))

    def _wkv(self, k, v, w, u):
        # stable RWKV-4 scan; k,v: (B,L,C); w,u: (C,)
        B, L, C = k.shape
        a = k.new_zeros(B, C); b = k.new_zeros(B, C); p = k.new_full((B, C), -1e30)
        outs = []
        for t in range(L):
            kt = k[:, t]; vt = v[:, t]
            q = torch.maximum(p, u + kt); e1 = torch.exp(p - q); e2 = torch.exp(u + kt - q)
            outs.append((e1 * a + e2 * vt) / (e1 * b + e2))
            q2 = torch.maximum(p + w, kt); e1 = torch.exp(p + w - q2); e2 = torch.exp(kt - q2)
            a = e1 * a + e2 * vt; b = e1 * b + e2; p = q2
        return torch.stack(outs, dim=1)

    def _scan(self, x, tag):
        mk = getattr(self, f"mix_k_{tag}"); mv = getattr(self, f"mix_v_{tag}"); mr = getattr(self, f"mix_r_{tag}")
        xs = F.pad(x, (0, 0, 1, 0))[:, :-1]                              # token shift (prev token)
        k = getattr(self, f"Wk_{tag}")(x * mk + xs * (1 - mk))
        v = getattr(self, f"Wv_{tag}")(x * mv + xs * (1 - mv))
        r = torch.sigmoid(getattr(self, f"Wr_{tag}")(x * mr + xs * (1 - mr)))
        w = -torch.exp(getattr(self, f"time_decay_{tag}")); u = getattr(self, f"time_first_{tag}")
        wkv = self._wkv(k, v, w, u)
        return getattr(self, f"Wo_{tag}")(r * wkv)

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            xf = x.float()
            y = self._scan(xf, "f")
            if self.bi: y = y + torch.flip(self._scan(torch.flip(xf, [1]), "b"), [1])
        return y.to(x.dtype)


def make_mixer(name, dim, n_heads, n_basis, n_states=64):
    if name == "yb":          return SpectralYBMixer(dim, n_basis=n_basis, orthogonal=True)
    if name == "yb_relaxed":  return SpectralYBMixer(dim, n_basis=n_basis, orthogonal=False)
    if name == "transformer": return AttentionMixer(dim, n_heads=n_heads)
    if name == "s4d":         return S4DMixer(dim, bidirectional=True)        # untuned (ablation)
    if name == "s4dlin":      return S4DLinMixer(dim, n_states=n_states, bidirectional=True)  # HiPPO/S4D-Lin (fair)
    if name == "lru":         return LRUMixer(dim, bidirectional=True)        # Linear Recurrent Unit
    if name == "fnet":        return FNetMixer(dim)                            # fixed-FFT cousin
    if name == "mamba":       return MambaMixer(dim, bidirectional=True)
    if name == "rglru":       return RGLRUMixer(dim, bidirectional=True)       # Hawk/Griffin RG-LRU
    if name == "scornn":      return SCORNNMixer(dim, bidirectional=True)      # scaled-Cayley orthogonal RNN
    if name == "rwkv":        return RWKVMixer(dim, bidirectional=True)        # RWKV-4 time mixing
    raise ValueError(name)


class Block(nn.Module):
    def __init__(self, name, dim, n_heads, n_basis, mlp_ratio, dropout, n_states=64):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.mixer = make_mixer(name, dim, n_heads, n_basis, n_states)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, mlp_ratio, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.mixer(self.norm1(x)))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class SequenceClassifier(nn.Module):
    """Shared scaffold: embed -> [Block]*depth -> mean-pool -> linear head."""
    def __init__(self, name, *, in_type, vocab, n_classes, seq_len,
                 dim=128, depth=4, n_heads=4, n_basis=24, mlp_ratio=2,
                 dropout=0.0, pos_emb=True, n_states=64):
        super().__init__()
        self.in_type = in_type
        if in_type == "discrete":
            self.embed = nn.Embedding(vocab, dim)
        else:  # continuous scalar per position
            self.embed = nn.Linear(1, dim)
        self.pos = nn.Parameter(0.02 * torch.randn(seq_len, dim)) if pos_emb else None
        self.blocks = nn.ModuleList([Block(name, dim, n_heads, n_basis, mlp_ratio, dropout, n_states)
                                     for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x):
        if self.in_type == "discrete":
            h = self.embed(x)                          # (B,L,dim)
        else:
            h = self.embed(x.unsqueeze(-1).float())    # (B,L,1)->(B,L,dim)
        if self.pos is not None:
            h = h + self.pos[: h.shape[1]].unsqueeze(0)
        for blk in self.blocks:
            h = blk(h)
        h = self.norm(h).mean(dim=1)
        return self.head(h)

    def forward_tokens(self, x):                  # per-position logits (for copy/seq-output tasks)
        if self.in_type == "discrete":
            h = self.embed(x)
        else:
            h = self.embed(x.unsqueeze(-1).float())
        if self.pos is not None:
            h = h + self.pos[: h.shape[1]].unsqueeze(0)
        for blk in self.blocks:
            h = blk(h)
        return self.head(self.norm(h))            # (B, L, n_classes)


def build_model(name, meta, dim, depth, n_heads=4, n_basis=24, dropout=0.0, pos_emb=True, n_states=64):
    return SequenceClassifier(
        name, in_type=meta["in_type"], vocab=meta.get("vocab", 0),
        n_classes=meta["n_classes"], seq_len=meta["seq_len"],
        dim=dim, depth=depth, n_heads=n_heads, n_basis=n_basis,
        dropout=dropout, pos_emb=pos_emb, n_states=n_states)


## 3 · Data loaders  (`%%writefile data.py`)

In [ ]:
%%writefile data.py
"""
Data loaders for the YB-Mixer harness. All tasks are long-range BIDIRECTIONAL classification.

  smoke   : random tensors (no download) -- for sanity tests on CPU
  pmnist  : permuted sequential MNIST  (L=784, continuous, 10 classes)
  smnist  : sequential MNIST           (L=784, continuous, 10 classes)
  scifar  : sequential CIFAR-10 (gray) (L=1024, continuous, 10 classes)  == LRA-Image
  imdb    : byte-level IMDB sentiment  (L=cfg, discrete vocab 256, 2 classes) ~ LRA-Text

Returns (train_loader, val_loader, meta) with
  meta = {seq_len, in_type in {continuous, discrete}, vocab, n_classes}.
"""
import torch
from torch.utils.data import TensorDataset, DataLoader


def _loaders(Xtr, ytr, Xva, yva, bs, workers, pin):
    pw = workers > 0
    tr = DataLoader(TensorDataset(Xtr, ytr), batch_size=bs, shuffle=True,
                    num_workers=workers, pin_memory=pin, drop_last=True, persistent_workers=pw)
    va = DataLoader(TensorDataset(Xva, yva), batch_size=bs, shuffle=False,
                    num_workers=workers, pin_memory=pin, persistent_workers=pw)
    return tr, va


def get_smoke(bs=8, workers=0, pin=False, in_type="continuous", L=64, n_classes=4, n=64):
    if in_type == "discrete":
        Xtr = torch.randint(0, 256, (n, L)); Xva = torch.randint(0, 256, (n, L))
        vocab = 256
    else:
        Xtr = torch.rand(n, L); Xva = torch.rand(n, L); vocab = 0
    ytr = torch.randint(0, n_classes, (n,)); yva = torch.randint(0, n_classes, (n,))
    tr, va = _loaders(Xtr, ytr, Xva, yva, bs, workers, pin)
    return tr, va, {"seq_len": L, "in_type": in_type, "vocab": vocab, "n_classes": n_classes}


def _mnist(root, permute, seed):
    from torchvision import datasets, transforms
    tf = transforms.ToTensor()
    tr = datasets.MNIST(root, train=True, download=True, transform=tf)
    te = datasets.MNIST(root, train=False, download=True, transform=tf)
    Xtr = tr.data.float().div(255.).view(-1, 784); ytr = tr.targets
    Xte = te.data.float().div(255.).view(-1, 784); yte = te.targets
    if permute:
        g = torch.Generator().manual_seed(seed); perm = torch.randperm(784, generator=g)
        Xtr = Xtr[:, perm]; Xte = Xte[:, perm]
    return Xtr, ytr, Xte, yte


def get_pmnist(root="./data", bs=64, workers=4, pin=True, seed=0):
    Xtr, ytr, Xte, yte = _mnist(root, permute=True, seed=seed)
    tr, va = _loaders(Xtr, ytr, Xte, yte, bs, workers, pin)
    return tr, va, {"seq_len": 784, "in_type": "continuous", "vocab": 0, "n_classes": 10}


def get_smnist(root="./data", bs=64, workers=4, pin=True, seed=0):
    Xtr, ytr, Xte, yte = _mnist(root, permute=False, seed=seed)
    tr, va = _loaders(Xtr, ytr, Xte, yte, bs, workers, pin)
    return tr, va, {"seq_len": 784, "in_type": "continuous", "vocab": 0, "n_classes": 10}


def get_scifar(root="./data", bs=64, workers=4, pin=True):
    from torchvision import datasets
    tr = datasets.CIFAR10(root, train=True, download=True)
    te = datasets.CIFAR10(root, train=False, download=True)
    def gray(d):
        x = torch.tensor(d.data).float().div(255.)          # (N,32,32,3)
        x = (0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2])  # luminance
        return x.view(-1, 1024)
    Xtr, ytr = gray(tr), torch.tensor(tr.targets)
    Xte, yte = gray(te), torch.tensor(te.targets)
    tr_, va_ = _loaders(Xtr, ytr, Xte, yte, bs, workers, pin)
    return tr_, va_, {"seq_len": 1024, "in_type": "continuous", "vocab": 0, "n_classes": 10}


def get_imdb(bs=32, workers=4, pin=True, L=1024, n_train=None):
    from datasets import load_dataset
    ds = load_dataset("imdb")
    def enc(split):
        texts = ds[split]["text"]; labels = ds[split]["label"]
        if split == "train" and n_train:
            texts, labels = texts[:n_train], labels[:n_train]
        X = torch.zeros(len(texts), L, dtype=torch.long)
        for i, t in enumerate(texts):
            b = t.encode("utf-8", "ignore")[:L]
            X[i, : len(b)] = torch.tensor(list(b), dtype=torch.long)
        return X, torch.tensor(labels)
    Xtr, ytr = enc("train"); Xte, yte = enc("test")
    tr, va = _loaders(Xtr, ytr, Xte, yte, bs, workers, pin)
    return tr, va, {"seq_len": L, "in_type": "discrete", "vocab": 256, "n_classes": 2}



# ----------------------------- ListOps (LRA) -----------------------------
# Tokens: 0-9 digits; 10='[MIN' 11='[MAX' 12='[MED' 13='[SM'(sum mod 10); 14=']'; 15=PAD
import numpy as _np
def _listops_expr(rng, depth, max_depth, max_args):
    leaf_p = 0.25 + 0.65 * depth / max_depth        # likelier to stop deeper -> bounded length
    if depth >= max_depth or rng.random() < leaf_p:
        v = int(rng.integers(0, 10)); return [v], v
    op = int(rng.integers(0, 4)); n = int(rng.integers(2, max_args + 1))
    toks = [10 + op]; vals = []
    for _ in range(n):
        t, v = _listops_expr(rng, depth + 1, max_depth, max_args); toks += t; vals.append(v)
    toks.append(14)
    if   op == 0: val = min(vals)
    elif op == 1: val = max(vals)
    elif op == 2: val = int(_np.median(_np.array(vals)))      # floor of median
    else:         val = sum(vals) % 10
    return toks, val

def _listops_set(n, max_len, max_depth, max_args, seed):
    rng = _np.random.default_rng(seed); X = []; y = []
    while len(X) < n:
        toks, val = _listops_expr(rng, 0, max_depth, max_args)
        if 4 <= len(toks) <= max_len:
            toks = toks + [15] * (max_len - len(toks)); X.append(toks); y.append(val)
    return torch.tensor(X), torch.tensor(y)

def get_listops(bs=32, workers=2, pin=True, max_len=1024, n_train=8000, n_val=2000,
                max_depth=5, max_args=4, seed=0):
    Xtr, ytr = _listops_set(n_train, max_len, max_depth, max_args, seed)
    Xva, yva = _listops_set(n_val, max_len, max_depth, max_args, seed + 1)
    tr, va = _loaders(Xtr, ytr, Xva, yva, bs, workers, pin)
    return tr, va, {"seq_len": max_len, "in_type": "discrete", "vocab": 16, "n_classes": 10}

# ------------------- Pathfinder-style synthetic connectivity -------------------
# NOTE: a self-contained REIMPLEMENTATION of Pathfinder's connectivity challenge
# (binary: are the two endpoints joined by a path among distractors), NOT the official
# pixel dataset; numbers are not directly comparable to published LRA-Pathfinder.
def _rand_walk(rng, G, start, steps):
    path=[start]; y,x=start
    for _ in range(steps):
        y=int(_np.clip(y+rng.integers(-1,2),0,G-1)); x=int(_np.clip(x+rng.integers(-1,2),0,G-1)); path.append((y,x))
    return path

def _guided_walk(rng, G, a, b, maxsteps):
    """Walk from a toward b with noise; returns path and whether it reached b."""
    path=[a]; y,x=a
    for _ in range(maxsteps):
        sy=_np.sign(b[0]-y); sx=_np.sign(b[1]-x)
        y=int(_np.clip(y+(sy if rng.random()<0.8 else rng.integers(-1,2)),0,G-1))
        x=int(_np.clip(x+(sx if rng.random()<0.8 else rng.integers(-1,2)),0,G-1))
        path.append((y,x))
        if (y,x)==b: return path,True
    return path,False

def _mark(grid,cells,val,G,thick=True):
    for (y,x) in cells:
        grid[y,x]=val
        if thick:
            for dy,dx in ((0,1),(1,0)):
                yy,xx=min(y+dy,G-1),min(x+dx,G-1); 
                if grid[yy,xx]==0: grid[yy,xx]=val

def _pathfinder_one(rng, G):
    grid=_np.zeros((G,G),dtype=_np.float32)
    connected=bool(rng.integers(0,2))
    p1=(int(rng.integers(0,G)),int(rng.integers(0,G)))
    p2=(int(rng.integers(0,G)),int(rng.integers(0,G)))
    while abs(p1[0]-p2[0])+abs(p1[1]-p2[1])<G//2:
        p2=(int(rng.integers(0,G)),int(rng.integers(0,G)))
    if connected:
        path,_=_guided_walk(rng,G,p1,p2,G*4); _mark(grid,path,0.5,G)
    else:
        mid=(int(rng.integers(0,G)),int(rng.integers(0,G)))
        path,_=_guided_walk(rng,G,p1,mid,G*2); _mark(grid,path,0.5,G)
        _mark(grid,_rand_walk(rng,G,p2,3),0.5,G)        # short isolated stub at p2
    for _ in range(rng.integers(1,3)):                  # few distractors
        s=(int(rng.integers(0,G)),int(rng.integers(0,G))); _mark(grid,_rand_walk(rng,G,s,G//3),0.5,G)
    grid[p1]=1.0; grid[p2]=1.0
    return grid.reshape(-1), int(connected)

def _pathfinder_set(n, G, seed):
    rng = _np.random.default_rng(seed); X = []; y = []
    for _ in range(n):
        g, lab = _pathfinder_one(rng, G); X.append(g); y.append(lab)
    return torch.tensor(_np.stack(X)), torch.tensor(y)

def get_pathfinder(bs=32, workers=2, pin=True, G=32, n_train=8000, n_val=2000, seed=0):
    Xtr, ytr = _pathfinder_set(n_train, G, seed); Xva, yva = _pathfinder_set(n_val, G, seed + 1)
    tr, va = _loaders(Xtr, ytr, Xva, yva, bs, workers, pin)
    return tr, va, {"seq_len": G * G, "in_type": "continuous", "vocab": 0, "n_classes": 2}

# ----------------------------- Induction Heads -----------------------------
# Predict the token immediately following a unique SPECIAL marker (token V-1).
# Data tokens are 0..V-2 (=> n_classes = V-1). Supports eval at longer L (extrapolation).
def _induction_set(n, L, V, seed):
    rng = _np.random.default_rng(seed)
    X = rng.integers(0, V - 1, size=(n, L))                  # data tokens 0..V-2
    pos = rng.integers(0, L - 1, size=n)                     # marker position (p+1 valid)
    y = X[_np.arange(n), pos + 1].copy()                     # target = token after marker
    X[_np.arange(n), pos] = V - 1                            # place unique SPECIAL marker
    return torch.tensor(X), torch.tensor(y)

def get_induction(bs=32, workers=2, pin=True, L=256, V=16, n_train=8000, n_val=2000, seed=0):
    Xtr, ytr = _induction_set(n_train, L, V, seed); Xva, yva = _induction_set(n_val, L, V, seed + 1)
    tr, va = _loaders(Xtr, ytr, Xva, yva, bs, workers, pin)
    # pos_emb must be OFF at build time for length extrapolation (see notebook).
    return tr, va, {"seq_len": L, "in_type": "discrete", "vocab": V, "n_classes": V - 1}

def induction_eval_set(L, V=16, n=2000, seed=123):
    """Return (X, y) tensors for evaluating induction-heads extrapolation at length L."""
    return _induction_set(n, L, V, seed)

# ----------------------------- Selective Copying -----------------------------
# Scatter n_data data tokens (1..V-2) among noise (0) in the first L-n_data slots; the last
# n_data slots are QUERY markers (V-1) where the model must emit the data tokens IN ORDER.
# Uses per-position outputs (SequenceClassifier.forward_tokens) + masked CE at the query slots.
def selective_copy_batch(B, L, V, n_data, seed=None):
    rng = _np.random.default_rng(seed)
    content = L - n_data
    X = _np.zeros((B, L), dtype=_np.int64)                   # noise = 0
    Y = _np.zeros((B, n_data), dtype=_np.int64)
    for i in range(B):
        vals = rng.integers(1, V - 1, size=n_data)           # data tokens 1..V-2
        posn = _np.sort(rng.choice(content, n_data, replace=False))
        X[i, posn] = vals; Y[i] = vals
    X[:, content:] = V - 1                                    # QUERY markers
    out_pos = torch.arange(content, L)
    return torch.tensor(X), torch.tensor(Y), out_pos

def get_data(task, bs, workers, pin, seq_len=1024):
    if task == "pmnist": return get_pmnist(bs=bs, workers=workers, pin=pin)
    if task == "smnist": return get_smnist(bs=bs, workers=workers, pin=pin)
    if task == "scifar": return get_scifar(bs=bs, workers=workers, pin=pin)
    if task == "imdb":   return get_imdb(bs=bs, workers=workers, pin=pin, L=seq_len)
    if task == "listops":    return get_listops(bs=bs, workers=workers, pin=pin, max_len=seq_len)
    if task == "pathfinder": return get_pathfinder(bs=bs, workers=workers, pin=pin)
    if task == "induction":  return get_induction(bs=bs, workers=workers, pin=pin, L=min(seq_len,256))
    raise ValueError(task)


## 4 · Training, multi-seed, and throughput

In [ ]:
import time, math, importlib, numpy as np, torch, torch.nn as nn
import models, data as datamod
importlib.reload(models); importlib.reload(datamod)
from models import build_model
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k): return x
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; DATA_ROOT='/kaggle/working/data'
META={'pmnist':{'seq_len':784,'in_type':'continuous','vocab':0,'n_classes':10},
      'smnist':{'seq_len':784,'in_type':'continuous','vocab':0,'n_classes':10},
      'scifar':{'seq_len':1024,'in_type':'continuous','vocab':0,'n_classes':10},
      'imdb':{'seq_len':1024,'in_type':'discrete','vocab':256,'n_classes':2},
      'listops':{'seq_len':1024,'in_type':'discrete','vocab':16,'n_classes':10},
      'pathfinder':{'seq_len':1024,'in_type':'continuous','vocab':0,'n_classes':2},
      'induction':{'seq_len':256,'in_type':'discrete','vocab':16,'n_classes':15}}

def _cos(step,total,warm,base):
    if step<warm: return base*step/max(1,warm)
    p=(step-warm)/max(1,total-warm); return 0.5*base*(1+math.cos(math.pi*min(1.0,p)))

@torch.no_grad()
def evaluate(model,loader):
    model.eval(); c=t=0
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE); c+=(model(x).argmax(1)==y).sum().item(); t+=y.numel()
    return c/max(1,t)

def _data(task,bs,workers,pin,seq_len):
    if task=='pmnist': return datamod.get_pmnist(root=DATA_ROOT,bs=bs,workers=workers,pin=pin)
    if task=='smnist': return datamod.get_smnist(root=DATA_ROOT,bs=bs,workers=workers,pin=pin)
    if task=='scifar': return datamod.get_scifar(root=DATA_ROOT,bs=bs,workers=workers,pin=pin)
    if task=='imdb':   return datamod.get_imdb(bs=bs,workers=workers,pin=pin,L=seq_len)
    if task=='listops':    return datamod.get_listops(bs=bs,workers=workers,pin=pin,max_len=seq_len)
    if task=='pathfinder': return datamod.get_pathfinder(bs=bs,workers=workers,pin=pin)
    if task=='induction':  return datamod.get_induction(bs=bs,workers=workers,pin=pin,L=min(seq_len,256))
    raise ValueError(task)

def train_one(task,model_name,dim=128,depth=4,epochs=40,bs=64,lr=3e-3,wd=0.05,dropout=0.1,
              n_basis=32,n_states=64,seq_len=1024,warmup_frac=0.1,grad_clip=1.0,label_smoothing=0.1,
              seed=0,amp=True,workers=2,pos_emb=True,return_model=False,verbose=True):
    torch.manual_seed(seed); amp=amp and DEVICE=='cuda'; pin=DEVICE=='cuda'
    tr,va,meta=_data(task,bs,workers,pin,seq_len)
    model=build_model(model_name,meta,dim=dim,depth=depth,n_basis=n_basis,n_states=n_states,dropout=dropout,pos_emb=pos_emb).to(DEVICE)
    npar=sum(p.numel() for p in model.parameters())
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=wd)
    scaler=torch.amp.GradScaler('cuda',enabled=amp); crit=nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    total=len(tr)*epochs; warm=int(warmup_frac*total); step=0; best=0.0; t0=time.time()
    if verbose: print(f'[{model_name} on {task} seed{seed}] params={npar:,} L={meta["seq_len"]} amp={amp}')
    for ep in range(epochs):
        model.train(); run=0.0; nb=0
        for x,y in tqdm(tr,leave=False,desc=f'{model_name}/{task}/s{seed} ep{ep}'):
            x,y=x.to(DEVICE),y.to(DEVICE)
            for g in opt.param_groups: g['lr']=_cos(step,total,warm,lr)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE,enabled=amp): loss=crit(model(x),y)
            scaler.scale(loss).backward()
            if grad_clip: scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),grad_clip)
            scaler.step(opt); scaler.update(); run+=loss.item(); nb+=1; step+=1
        acc=evaluate(model,va); best=max(best,acc)
        if verbose and (ep%5==0 or ep==epochs-1):
            print(f'  ep{ep:3d} loss={run/max(1,nb):.4f} val_acc={acc:.4f} best={best:.4f} ({time.time()-t0:.0f}s)')
    res={'task':task,'model':model_name,'params':npar,'best_val_acc':best,'epochs':epochs,
         'dim':dim,'depth':depth,'seed':seed,'seconds':round(time.time()-t0,1)}
    return (res, model) if return_model else res

def train_seeds(task,model_name,seeds=(0,1,2),**kw):
    accs=[]; last=None
    for sd in seeds:
        r=train_one(task,model_name,seed=sd,verbose=(sd==seeds[0]),**kw); accs.append(r['best_val_acc']); last=r
        print(f'   {model_name}/{task} seed {sd}: {r["best_val_acc"]:.4f}')
    out={k:last[k] for k in ('task','model','params','epochs','dim','depth')}
    out.update(seeds=list(seeds),accs=accs,acc_mean=float(np.mean(accs)),acc_std=float(np.std(accs)))
    print(f'   => {model_name}/{task}: {out["acc_mean"]:.4f} ± {out["acc_std"]:.4f}')
    return out

@torch.no_grad()
def throughput(model_name,task='scifar',dim=128,depth=6,bs=16,reps=20):
    meta=META[task]; net=build_model(model_name,meta,dim=dim,depth=depth).to(DEVICE).eval()
    if meta['in_type']=='discrete': x=torch.randint(0,256,(bs,meta['seq_len']),device=DEVICE)
    else: x=torch.rand(bs,meta['seq_len'],device=DEVICE)
    for _ in range(3): net(x)
    if DEVICE=='cuda': torch.cuda.synchronize()
    t0=time.time()
    for _ in range(reps): net(x)
    if DEVICE=='cuda': torch.cuda.synchronize()
    dt=(time.time()-t0)/reps
    return {'model':model_name,'task':task,'ms_per_batch':round(dt*1e3,2),'seq_per_s':round(bs/dt,1)}


## 4b · Model size presets (param scaling)`large` crosses **5M params** for every model (W2: scale). Params reported for sCIFAR.

In [ ]:
PRESETS = {'base':  dict(dim=128, depth=6, bs=64),   # ~0.5-2M  (cheap; good for multi-seed)
           'mid':   dict(dim=256, depth=8, bs=48),   # ~2-4M
           'large': dict(dim=384, depth=8, bs=32)}   # >5M : yb~5.3M s4dlin~6.3M lru~5.2M transformer~9.9M
def show_params(size, task='scifar', mods=('yb','s4dlin','lru','fnet','transformer','mamba')):
    cfg=PRESETS[size]; meta=META[task]
    print(f"size='{size}'  dim={cfg['dim']} depth={cfg['depth']} bs={cfg['bs']}")
    for m in mods:
        n=sum(p.numel() for p in build_model(m,meta,dim=cfg['dim'],depth=cfg['depth']).parameters())
        print(f'  {m:<12} {n/1e6:6.2f}M')
show_params('large')


## 5 · Smoke test (≈15 s) — every mixer incl. the new fair baselines

In [ ]:
for m in ['yb','yb_relaxed','fnet','lru','s4dlin','s4d','transformer','mamba']:
    r=train_one('pmnist',m,dim=32,depth=2,epochs=1,bs=128,workers=2,verbose=False)
    print(f"{m:<12} OK  params={r['params']:,}  1-epoch acc={r['best_val_acc']:.3f}")


### Sequential-recurrence baselines (RG-LRU / scoRNN / RWKV) — functional checkThese use a **sequential scan** (no FFT fast path), so they are the slowest mixers. Quick forward/grad check at small L only; see Phase C for training.

In [ ]:
import torch as _t
for m in ['rglru','scornn','rwkv']:
    net=build_model(m,{'seq_len':64,'in_type':'continuous','vocab':0,'n_classes':4},dim=48,depth=2).to(DEVICE)
    x=_t.rand(4,64,device=DEVICE); y=net(x); y.sum().backward()
    print(f"{m:<7} OK  params={sum(p.numel() for p in net.parameters()):,}")


## 6 · Results collector

In [ ]:
import json, pandas as pd
RESULTS=[]
def log(r):
    RESULTS.append(r); json.dump(RESULTS,open('/kaggle/working/results.json','w'),indent=2)
    return pd.DataFrame(RESULTS)
def table():
    rows=[]
    for r in RESULTS:
        acc = (f"{r['acc_mean']:.4f} ± {r['acc_std']:.4f}" if 'acc_mean' in r else f"{r['best_val_acc']:.4f}")
        rows.append({'task':r['task'],'model':r['model'],'params':r['params'],
                     'acc':acc,'epochs':r['epochs']})
    return pd.DataFrame(rows).sort_values(['task','model'])


## 7 · BUDGET NOTE — read before Phase A
`#runs = len(MODELS) × len(SEEDS)`; cost also scales with the **size preset**.
- **Fair comparison, multi-seed (do this first):** `SIZE='base'`, `MODELS=['yb','s4dlin','lru','fnet']`, `SEEDS=[0,1,2]`, `EPOCHS=50`  → fits ~12 h on one T4. Nails W1/W7.
- **Scale to >5M (W2):** `SIZE='large'` is ~10× the FLOPs — **one run can take a few hours**. Use `SEEDS=[0]` (or `[0,1]`) and `EPOCHS≈30–40`, models `['yb','s4dlin','lru']`. That is Phase A-Large below.
- You likely **cannot** do both fully in one 12 h session on a single T4. Recommended split: multi-seed `base` for fairness **+** single-seed `large` for the scale headline. Never drop `s4dlin`/`lru`.

## 8 · Phase A — sequential-CIFAR (LRA-Image), the contested result, MULTI-SEED

In [ ]:
SIZE   = 'base'                       # 'base' (multi-seed) | 'mid' | 'large' (>5M)
MODELS = ['yb','s4dlin','lru','fnet'] # add 'transformer','mamba','s4d' if budget allows
SEEDS  = [0,1,2]                      # for SIZE='large' use [0] or [0,1]
EPOCHS = 50                           # for SIZE='large' try 30-40
cfg = PRESETS[SIZE]
for m in MODELS:
    log(train_seeds('scifar', m, seeds=SEEDS, dim=cfg['dim'], depth=cfg['depth'],
                    epochs=EPOCHS, bs=cfg['bs'], lr=3e-3, dropout=0.1))
table()


## 8b · Phase A-Large — **>5M params** on sequential-CIFAR (addresses scale, W2)Single-seed by default to fit the budget; bump `SEEDS` if you have a faster GPU. Uses `bs=32`.

In [ ]:
cfg = PRESETS['large']                 # dim=384, depth=8  -> >5M params
print('param check:'); show_params('large', mods=('yb','s4dlin','lru'))
for m in ['yb','s4dlin','lru']:        # core fair trio at >5M
    log(train_seeds('scifar', m, seeds=[0], dim=cfg['dim'], depth=cfg['depth'],
                    epochs=40, bs=cfg['bs'], lr=2e-3, dropout=0.1))
table()


## 9 · Phase B — permuted-MNIST & byte-IMDB (multi-seed, optional)

In [ ]:
# permuted-MNIST
for m in ['yb','s4dlin','lru']:
    log(train_seeds('pmnist', m, seeds=[0,1,2], dim=128, depth=4, epochs=40, bs=64, lr=3e-3))
# byte-level IMDB (Internet ON)
for m in ['yb','s4dlin','lru']:
    log(train_seeds('imdb', m, seeds=[0,1,2], dim=128, depth=4, epochs=25, seq_len=1024, bs=32, lr=2e-3))
table()


## 9b · Phase C — sequential-recurrence baselines (SLOW: RG-LRU / scoRNN / RWKV)
These scan the sequence step-by-step in PyTorch, so at L=1024 they are **far** slower than the
FFT-based mixers (consistent with the Griffin/scoRNN papers, which need custom kernels). Run them
**single-seed, base size, fewer epochs**, and consider a shorter task first. Do **not** put them in
the multi-seed sweep. If too slow on sCIFAR, run on pMNIST or reduce `EPOCHS`.

In [ ]:
SEQ_MODELS = ['rglru','scornn','rwkv']
SEQ_TASK   = 'scifar'      # 'pmnist' is shorter (L=784) if sCIFAR is too slow
cfg = PRESETS['base']
for m in SEQ_MODELS:
    log(train_seeds(SEQ_TASK, m, seeds=[0], dim=cfg['dim'], depth=cfg['depth'],
                    epochs=30, bs=cfg['bs'], lr=3e-3, dropout=0.1))
table()


## 9c · Phase D — extended downstream tasks (ListOps · Induction · Pathfinder)ListOps and Induction validated as learnable; Pathfinder here is a **synthetic** reimplementation (hard tracing task, *not* comparable to official LRA-Pathfinder) — treat as experimental.

In [ ]:
# ListOps (LRA) and Induction Heads (induction built with pos_emb=False for extrapolation)
for m in ['yb','s4dlin','lru']:
    log(train_seeds('listops', m, seeds=[0], dim=128, depth=6, epochs=40, bs=32, lr=2e-3))
for m in ['yb','s4dlin','lru']:
    log(train_seeds('induction', m, seeds=[0], dim=128, depth=4, epochs=25, bs=64, lr=3e-3, pos_emb=False))
# Pathfinder-style (experimental, synthetic) — may need many epochs to break chance
log(train_seeds('pathfinder','yb', seeds=[0], dim=128, depth=6, epochs=40, bs=32, lr=2e-3))
table()


### Induction Heads — length extrapolation (train L=256, eval longer)Recurrent/structured mixers should extrapolate to L>train; this is the retrieval+extrapolation story from the Griffin paper (Fig. 6b).

In [ ]:
import data as datamod, matplotlib.pyplot as plt
def induction_extrapolation(model_name='yb', dim=128, depth=4, epochs=25, train_L=256,
                            eval_Ls=(256,512,1024,2048)):
    res, net = train_one('induction', model_name, dim=dim, depth=depth, epochs=epochs, bs=64,
                         lr=3e-3, pos_emb=False, seq_len=train_L, return_model=True, verbose=False)
    net.eval(); accs=[]
    for L in eval_Ls:
        Xe,ye=datamod.induction_eval_set(L)
        with torch.no_grad():
            p=[]
            for i in range(0,len(Xe),256):
                p.append(net(Xe[i:i+256].to(DEVICE)).argmax(1).cpu())
            acc=(torch.cat(p)==ye).float().mean().item()
        accs.append(acc); print(f'  {model_name}  eval L={L:>5}  acc={acc:.3f}')
    return eval_Ls, accs

plt.figure(figsize=(5,3.2))
for m in ['yb','s4dlin','lru']:
    Ls, accs = induction_extrapolation(m, epochs=25)
    plt.plot(Ls, accs, 'o-', label=m)
plt.axvline(256, ls='--', c='gray', lw=1, label='train length')
plt.xscale('log', base=2); plt.xlabel('eval sequence length'); plt.ylabel('induction acc')
plt.title('Induction Heads: length extrapolation'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig('/kaggle/working/induction_extrapolation.png', dpi=120); plt.show()


## 9d · Selective Copying (per-position output via forward_tokens)Copy `n_data` data tokens scattered among noise, emitted in order at the query slots. Trained with masked cross-entropy at the output positions.

In [ ]:
import data as datamod, torch.nn as nn
def train_selective_copy(model_name='yb', V=16, L=1024, n_data=16, dim=128, depth=6,
                         steps=2000, bs=64, lr=2e-3):
    meta={'seq_len':L,'in_type':'discrete','vocab':V,'n_classes':V}
    net=build_model(model_name,meta,dim=dim,depth=depth,pos_emb=True).to(DEVICE)
    opt=torch.optim.AdamW(net.parameters(),lr=lr); lf=nn.CrossEntropyLoss()
    net.train()
    for step in range(steps):
        X,Y,outp=datamod.selective_copy_batch(bs,L,V,n_data,seed=step)
        X,Y=X.to(DEVICE),Y.to(DEVICE)
        logits=net.forward_tokens(X)[:,outp]
        loss=lf(logits.reshape(-1,V),Y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    net.eval()
    Xv,Yv,outp=datamod.selective_copy_batch(1000,L,V,n_data,seed=999999)
    with torch.no_grad(): pred=net.forward_tokens(Xv.to(DEVICE))[:,outp].argmax(-1).cpu()
    tok=(pred==Yv).float().mean().item(); seq=(pred==Yv).all(1).float().mean().item()
    print(f'{model_name}: token_acc={tok:.3f}  seq_acc={seq:.3f}')
    return {'task':'selective_copy','model':model_name,'token_acc':tok,'seq_acc':seq}
for m in ['yb','s4dlin','lru']:
    train_selective_copy(m, steps=2000)


## 10 · Throughput (latency evidence for the efficiency claim)

In [ ]:
thr=[throughput(m,'scifar',dim=128,depth=6,bs=16) for m in ['yb','s4dlin','lru','fnet','transformer','mamba']]
import pandas as pd; tdf=pd.DataFrame(thr); display(tdf)
# sequential baselines are slow -> fewer reps, smaller batch
thr_seq=[throughput(m,'scifar',dim=128,depth=6,bs=8,reps=3) for m in ['rglru','scornn','rwkv']]
tdf=pd.concat([tdf,pd.DataFrame(thr_seq)],ignore_index=True); display(tdf)
tdf.to_csv('/kaggle/working/throughput.csv',index=False)


## 11 · Final tables

In [ ]:
import pandas as pd
df=table(); display(df)
df.to_csv('/kaggle/working/results_table.csv',index=False)
print('saved: /kaggle/working/results.json , results_table.csv , throughput.csv')
